In [1]:
import os, sys
sys.path.append(".")

import numpy as np
import torch
import torch.nn.functional as F
from torch.amp import autocast
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer, train_bpe
from storybot.utils import get_device, generate

In [2]:
def get_lr(it, max_lr, warmup_iters, max_iters):
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress) # 서서히 줄어들게
    return 0.0

def get_batch(data, context_len, batch_size, device, random = True, offset = 0): # 한꺼번에 몇개의 문맥을 처리할 것인가에 대한
    if random:
        ix = torch.randint(len(data) - context_len - 1, (batch_size,))
    else:
        ix = torch.arange(offset, offset + batch_size * context_len, context_len)
        ix = ix[ix + context_len + 1 < len(data)]
        if len(ix) == 0:
            return None, None

    x = torch.stack([torch.from_numpy(data[i:i + context_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+context_len+1].astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)

def evaluate(model, val_data, context_len, batch_size, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0.0

    max_start = len(val_data) - context_len - 1
    num_batches = (max_start // context_len) // batch_size + 1

    with torch.no_grad():
        for batch_idx in tqdm(range(num_batches), desc = "Validation"):
            offset = batch_idx * batch_size * context_len
            x, y = get_batch(val_data,
                             context_len,
                             batch_size,
                             device,
                             random = False,
                             offset = offset)
            if x is None:
                break
            with autocast(device_type = device.type, dtype = torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                       y.view(-1),
                                       reduction = "sum")
            total_loss += loss.item()
            total_tokens += y.numel()
    model.train()
    return total_loss / total_tokens

In [3]:
device = get_device()
print(device)

vocab_size = 10000
file_path = "storybot/tiny_stories_train.txt"

merge_rules = train_bpe(file_path, vocab_size, num_processes = 8)

with open("storybot/merge_rules.pkl", "wb") as f:
    pickle.dump(merge_rules, f)

cuda


Training BPE: 100%|██████████| 9743/9743 [00:36<00:00, 267.69it/s]


In [4]:
merge_rules_path = "storybot/merge_rules.pkl"
tokenizer = BPETokenizer.load_from(merge_rules_path)

tokenizer.encode_file("storybot/tiny_stories_train.txt",
                      "storybot/tiny_stories_train.bin",
                      num_processes = 8)

tokenizer.encode_file("storybot/tiny_stories_valid.txt",
                      "storybot/tiny_stories_valid.bin",
                      num_processes = 8)

Encoding chunks: 100%|██████████| 64/64 [00:04<00:00, 13.79it/s]


5465873

In [5]:
device = get_device()
print(device)
data_path = "storybot/tiny_stories_train.bin"
valid_data_path = "storybot/tiny_stories_valid.bin"
tokenizer_path = "storybot/merge_rules.pkl"
model_save_path = "storybot/model_pretrain.pt"

context_len = 256
vocab_size = 10000
batch_size = 32
learning_rate = 0.001
warmup_iters = 200
#max_iters = 40000
max_iters = 10000
embed_dim = 512
n_head = 16
n_layers = 4
ff_dim = 1344
theta = 10000
eval_iters = 500
grad_clip = 1.0
save_iters = [500, 5000]

train_data = np.memmap(data_path, dtype = np.uint16, mode = "r")
valid_data = np.memmap(valid_data_path, dtype = np.uint16, mode = "r")

tokenizer = BPETokenizer.load_from(tokenizer_path)

model = GPT(vocab_size,
            context_len,
            embed_dim,
            n_head,
            n_layers,
            ff_dim,
            theta).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

total_params = sum(p.numel() for p in model.parameters())
print(total_params)

pbar = tqdm(range(max_iters))

val_loss = float("inf")
val_losses = []
val_iters = []

cuda
22696448


  0%|          | 0/10000 [00:00<?, ?it/s]

In [ ]:
for i in pbar:
    lr = get_lr(i, learning_rate, warmup_iters, max_iters)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    batch_x, batch_y = get_batch(train_data, context_len, batch_size, device)
    optimizer.zero_grad()

    with autocast(device_type = device.type, dtype = torch.bfloat16):
        logits = model(batch_x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               batch_y.view(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if i in save_iters:
        save_path = f"storybot/model_iter_{i}.pt"
        model.save(save_path)
        print("모델이 저장되었습니다.")

    if (i % eval_iters) == 0 or i == max_iters -1:
        val_loss = evaluate(model, valid_data, context_len, batch_size, device)
        val_losses.append(val_loss)
        val_iters.append(i)

    pbar.set_postfix({"loss": f"{loss.item():.4f}",
                      "val_loss": f"{val_loss:.4f}"})

model.save(model_save_path)

  5%|▌         | 500/10000 [01:42<38:34,  4.10it/s, loss=2.4291, val_loss=9.3365] 

모델이 저장되었습니다.


 25%|██▌       | 2500/10000 [13:11<30:24,  4.11it/s, loss=1.8270, val_loss=1.8737]   

In [ ]:
import os, sys
sys.path.append('.')

from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

device = get_device()
model_path = "storybot/model_pretrain.pt"
tokenizer_path = "storybot/merge_rules.pkl"

prompt = "Once upon a time"
prompt = "<|endoftext|>"
max_new_tokens = 300
temperature = 1.0
num_samples = 3

tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT.load_from(model_path, device = device)

for i in range(num_samples):
    print(f"샘플 : {i + 1}")
    story = generate(model,
                     tokenizer,
                     prompt,
                     max_new_tokens,
                     temperature)
    print(story)

샘플 : 1

One day, a soft frog named Fifi lived in a small pond. Fifi was sad and miserable. She could not find any food to eat. Freddy wanted to eat in the big green pond, but he was too small to hop in the pond.
A duck named Ducky saw Froggy looking for food. Ducky asked, "Why are you so sad?" Froggy said, "I miss any food, can you show me how to find food?" Ducky smiled and said, "Sure, I can show you a new place to find! Follow me!"
Ducky and Froggy went to a new pond with lots of fish. They found yummy fish, tasty fish, and a big tree. Ducky and Froggy were so happy. They ate the fish and played in the pond all day. At the end of the day, they went back to the pond and had a home. Fifi thanked Ducky for helping her. From that day on, Fifi and Bunny became best friends. They always helped each other and had lots of fun.

샘플 : 2


Lucy noticed that the bottle was dark outside, so she went to take a look. She took the bottle and started to play. Suddenly, Lucy spilled so much liquid on

In [ ]:
# 샘플 1,2,3 이 얼마나 잘만들어졌는지 평가
import json
import statistics
from openai import OpenAI
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

client = OpenAI(api_key = "")

device = get_device()
tokenizer_path = "storybot/merge_rules.pkl"
tokenizer = BPETokenizer.load_from(tokenizer_path)

model_paths = {500 : "storybot/model_iter_500.pt",
               5000 : "storybot/model_iter_5000.pt",
               10000 : "storybot/model_pretrain.pt"}

prompt = "<|endoftext|>"
max_new_tokens = 200
temperature = 1.0
num_samples = 10

def evaluate_story(client, story):
    evaluation_prompt = f"""다음 어린이용 스토리를 두 가지 관점에서 1 ~ 5 점으로 평가해줘.

스토리 :
{story}

평가관점 :
1. Coherence(일관성) : 논리적으로 이어지는가, 이야기로서 앞뒤가 맞는가
2. Grammar(문법) : 문법적으로 올바른 영어인가

다음 JSON 형식으로 대답해줘.
{{"coherence": <1~5 범위의 정수>,
  "grammar": <1~5 범위의 정수>,
  "comment": <평가에 대한 간단한 이유>}}"""

    response = client.chat.completions.create(model = "gpt-4o-mini",
                                              messages = [{"role": "user",
                                                           "content" : evaluation_prompt}],
                                              max_tokens = 300,
                                              response_format = {"type": "json_object"})
    text = response.choices[0].message.content
    print(text)
    return json.loads(text)

results = {}
for iteration, model_path in model_paths.items():
    print(f"Iteration : {iteration}")

    model = GPT.load_from(model_path, device = device)
    iteration_results = []

    for i in range(num_samples):
        print(f"샘플 {i + 1}")
        story = generate(model,
                         tokenizer,
                         prompt,
                         max_new_tokens,
                         temperature)
        print(f"Story : {story[:200]}")

        scores = evaluate_story(client, story)
        print(f"Scores : {scores}")

        iteration_results.append({"story": story, "scores": scores})
    results[iteration] = iteration_results

print("Summary")

for iteration in model_paths.keys():
    scores_list = [r["scores"] for r in results[iteration]]
    print(f"Iteration {iteration}")
    for key in ["coherence", "grammar"]:
        values = [s[key] for s in scores_list]
        avg = statistics.mean(values)
        std = statistics.stdev(values) if len(values) > 1 else 0
        print(f"{key}: {avg:.2f} +- {std:.2f}")

Iteration : 500
샘플 1
Story : 

John was having lots of fun playing. He looked through his bed, he needed to be a rare one of what had happened. All his friends talked to their bucket and immediately said he was very friendly. The
{
  "coherence": 2,
  "grammar": 2,
  "comment": "The story lacks logical flow and clarity, making it difficult to follow. There are several grammatical errors and awkward phrasing that hinder comprehension."
}
Scores : {'coherence': 2, 'grammar': 2, 'comment': 'The story lacks logical flow and clarity, making it difficult to follow. There are several grammatical errors and awkward phrasing that hinder comprehension.'}
샘플 2
Story : 

Once upon a time, there was a little girl named Lily. Lily had left her room and they forgot to play. Lily was a most attractive doll on the couch and her daughter because the special family had shu
{"coherence": 2, "grammar": 2, "comment": "The story lacks logical flow and clarity, making it difficult to follow. Additionally, the

In [ ]:
# 직접선호도최적화
import sys
sys.path.append(".")
from itertools import cycle
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device

In [ ]:
device = get_device()
data_path = "storybot/tiny_stories_dpo.json"
tokenizer_path = "storybot/merge_rules.pkl"
pretrain_model_path = "storybot/model_pretrain.pt"
dpo_model_save_path = "storybot/model_dpo.pt"

context_len = 256
batch_size = 8
learning_rate = 5e-6
beta = 0.1
max_iters = 1000

In [ ]:
class DPODataset(Dataset):
    def __init__(self, data_path, tokenizer, context_len):
        self.tokenizer = tokenizer
        self.context_len = context_len
        self.samples = []

        with open(data_path) as f:
            data = json.load(f)

        for item in data:
            sample = self._create_sample(item["prompt"],
                                         item["chosen"],
                                         item["rejected"])
            self.samples.append(sample)

    def _pad_and_mask(self, ids, prompt_len):
        mask = [0] * prompt_len + [1] * (len(ids) - prompt_len)
        if len(ids) > self.context_len:
            ids = ids[:self.context_len]
            mask = mask[:self.context_len]
        else:
            pad_len = self.context_len - len(ids)
            ids = ids + [0] * pad_len
            mask = mask + [0] * pad_len
        return ids, mask

    def _create_sample(self, prompt, chosen, rejected):
        prompt_ids = self.tokenizer.encode(prompt)
        chosen_ids = prompt_ids + self.tokenizer.encode(chosen)
        rejected_ids = prompt_ids + self.tokenizer.encode(rejected)

        prompt_len = len(prompt_ids)
        chosen_ids, chosen_mask = self._pad_and_mask(chosen_ids, prompt_len)
        rejected_ids, rejected_mask = self._pad_and_mask(rejected_ids, prompt_len)

        return chosen_ids, chosen_mask, rejected_ids, rejected_mask

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        chosen_ids, chosen_mask, rejected_ids, rejected_mask = self.samples[idx]
        return(torch.tensor(chosen_ids, dtype = torch.long),
               torch.tensor(chosen_mask, dtype = torch.long),
               torch.tensor(rejected_ids, dtype = torch.long),
               torch.tensor(rejected_mask, dtype = torch.long))

def get_sequence_logprobs(model, ids, mask):
    logits = model(ids)
    log_probs = F.log_softmax(logits[:, :-1, :], dim = -1)
    labels = ids[:, 1:]

    per_token_logprobs = torch.gather(log_probs,
                                      dim = -1,
                                      index = labels.unsqueeze(-1)).squeeze(-1)
    masked_logprobs = per_token_logprobs * mask[:, 1:]
    return masked_logprobs.sum(dim = -1)

def compute_dpo_loss(model,
                     ref_model,
                     chosen_ids,
                     chosen_mask,
                     rejected_ids,
                     rejected_mask,
                     beta):
    chosen_logprobs = get_sequence_logprobs(model, chosen_ids, chosen_mask)
    rejected_logprobs = get_sequence_logprobs(model, rejected_ids, rejected_mask)

    with torch.no_grad():
        ref_chosen_logprobs = get_sequence_logprobs(ref_model, chosen_ids, chosen_mask)
        ref_rejected_logprobs = get_sequence_logprobs(ref_model, rejected_ids, rejected_mask)

    logits = beta * ((chosen_logprobs - rejected_logprobs) - (ref_chosen_logprobs - ref_rejected_logprobs))

    return -F.logsigmoid(logits).mean()

tokenizer = BPETokenizer.load_from(tokenizer_path)
dataset = DPODataset(data_path, tokenizer, context_len)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

model = GPT.load_from(pretrain_model_path, device = device)
ref_model = GPT.load_from(pretrain_model_path, device = device)
ref_model.eval()
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

losses = []
data_iter = cycle(dataloader)

In [ ]:
pbar = tqdm(range(max_iters))

for i in pbar:
    chosen_ids, chosen_mask, rejected_ids, rejected_mask = next(data_iter)
    chosen_ids, chosen_mask = chosen_ids.to(device), chosen_mask.to(device)
    rejected_ids, rejected_mask = rejected_ids.to(device), rejected_mask.to(device)

    loss = compute_dpo_loss(model,
                            ref_model,
                            chosen_ids,
                            chosen_mask,
                            rejected_ids,
                            rejected_mask,
                            beta)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({"loss": f"{loss.item():.4f}"})

model.save(dpo_model_save_path)

100%|██████████| 1000/1000 [02:04<00:00,  8.00it/s, loss=0.0026]


In [ ]:
import sys
sys.path.append(".")

import json
import torch
import torch.nn.functional as F
from openai import OpenAI
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

client = OpenAI(api_key = "")

device = get_device()
tokenizer_path = "storybot/merge_rules.pkl"
tokenizer = BPETokenizer.load_from(tokenizer_path)

model_paths = {"pretrain" : "storybot/model_pretrain.pt",
               "dpo" : "storybot/model_dpo.pt"}

prompt = "Once upon a time"
num_comparisons = 100
max_new_tokens = 150
temperature = 1.0

def compare_stories(client, story_a, story_b):
    """두 스토리를 비교하여 어느 쪽이 더 해피엔딩에 가까운지 판정"""
    evaluation_prompt = f"""다음의 두 어린이용 스토리를 비교하여, 어느 쪽이 더 해피엔딩인지 판단해줘.
    [Story A]
    {story_a}
    [Story B]
    {story_b}
    어느 쪽의 결말이 더 밝고 행복한지, 또는 내용이 더 희망적인지 판단해줘.
    JSON형식으로 답변 : {{"winner": "A" or "B" or "tie", "reason": "간단한 이유"}}"""

    response = client.chat.completions.create(model = "gpt-4o-mini",
                                              messages = [{"role": "user",
                                                           "content": evaluation_prompt}],
                                              max_tokens = 150,
                                              response_format = {"type": "json_object"})
    text = response.choices[0].message.content
    return json.loads(text)

model_pretrain = GPT.load_from(model_paths["pretrain"], device = device)
model_dpo = GPT.load_from(model_paths["dpo"], device= device)

results = []
wins = {"pretrain": 0, "dpo": 0, "tie": 0}

for i in range(num_comparisons):
    print(f"Comparison {i + 1} / {num_comparisons}")

    story_pretrain = generate(model_pretrain,
                              tokenizer,
                              prompt,
                              max_new_tokens,
                              temperature)
    story_dpo = generate(model_dpo,
                         tokenizer,
                         prompt,
                         max_new_tokens,
                         temperature)

    print(f"[Pretrain]: {story_pretrain[:100]}...")
    print(f"[dpo]: {story_dpo[:100]}...")

    import random

    if random.random() < 0.5:
        story_a, story_b = story_pretrain, story_dpo
        mapping = {"A": "pretrain", "B": "dpo"}
    else:
        story_a, story_b = story_dpo, story_pretrain
        mapping = {"A": "dpo", "B": "pretrain"}

    judgement = compare_stories(client, story_a, story_b)

    winner_label = judgement["winner"]
    if winner_label == "tie":
        winner = "tie"
    else:
        winner = mapping[winner_label]
    wins[winner] += 1

    print(f"승자: {winner}")
    print(f"이유: {judgement["reason"]}")

    results.append({"story_pretrain": story_pretrain,
                    "story_dpo": story_dpo,
                    "winner": winner,
                    "reason": judgement["reason"]})

print("일대일 비교 결과")

total = num_comparisons
print(f"사전 학습 모델 승리: {wins['pretrain']:3d} ({wins['pretrain']/total*100:5.1f}%)")
print(f"DPO 모델 승리: {wins['dpo']:3d} ({wins['dpo']/total*100:5.1f}%)")
print(f"무승부: {wins['tie']:3d} ({wins['tie']/total*100:5.1f}%)")

if wins['pretrain'] + wins['dpo'] > 0:
    dpo_winrate = wins['dpo'] / (wins['pretrain'] + wins['dpo']) * 100
    print(f"\n  DPO 모델 승률(무승부 제외): {dpo_winrate:.1f}%")

Comparison 1 / 100
[Pretrain]: Once upon a time, there was a big wolf. The wolf was not wild, but he was very smart. Every day, he ...
[dpo]: Once upon a time, there was a very flexible cat named Chirpy. Chirpy loved to play and have fun. One...
승자: pretrain
이유: 스스로 변화를 이룬 큰 늑대가 작은 소녀와의 만남을 통해 긍정적인 변화를 경험하고 친구가 되는 이야기로, 더 행복하고 희망적인 결말을 가지고 있다.
Comparison 2 / 100
[Pretrain]: Once upon a time, there was a little girl named Sue. Sue had a favorite jacket. This jacket was very...
[dpo]: Once upon a time, in a small town, there was a famous bird named Blue. Blue could sing very well. Al...
승자: dpo
이유: Story A ends with Blue and Tweetie helping a friend in need, while Story B ends with Sue feeling sad and not having a happy resolution.
Comparison 3 / 100
[Pretrain]: Once upon a time, in a small house, there was a huge chimney above Sara's house. Every day, the chim...
[dpo]: Once upon a time, there was a brave little dog named Spot. Spot loved to protect his friends in the ...
승자: pretrain
이